# 42 — Load the Saved MotionSense-AI Model and Predict on New Sensor Values

## Goal

To load the Trained Random Forest model, verify its accuracy, and use it to predict activities for **10 new sets of accelerometer and gyroscope values**.

## Prediction Pipeline

```text
Load Saved Model
        ↓
Load Processed Dataset
        ↓
Recreate the Same Test Split
        ↓
Measure Model Accuracy
        ↓
Enter 10 New Sensor Samples
        ↓
Engineer Magnitude Features
        ↓
Predict Activities
        ↓
Display a Clean Results Table
```

> **Important:** Run Notebook 41 first so that the trained model and processed dataset exist in the shared `../data/processed` folder.....

## Part 1 — Import Libraries and Define File Paths

This notebook uses:

- **Joblib** to load the saved model
- **Pandas** to organize data
- **NumPy** to calculate magnitude features
- **Scikit-learn** to recreate the test split and measure accuracy


In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

MODEL_PATH = PROCESSED_DIR / "motionsense_final_model.joblib"
ML_READY_PATH = PROCESSED_DIR / "motionsense_ml_ready.csv"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("Model path:", MODEL_PATH)
print("Dataset path:", ML_READY_PATH)


Model path: ..\data\processed\motionsense_final_model.joblib
Dataset path: ..\data\processed\motionsense_ml_ready.csv


## Part 2 — Confirm That Notebook 41 Outputs Exist

A clear error message is shown if the model or processed dataset is missing.


In [2]:
missing_files = [
    path for path in [MODEL_PATH, ML_READY_PATH]
    if not path.exists()
]

if missing_files:
    missing_text = "\n".join(f"• {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required files were not found:\n"
        f"{missing_text}\n\n"
        "Please run Notebook 41 completely before running this notebook."
    )

print("Required model and dataset files were found.")


Required model and dataset files were found.


## Part 3 — Load the Trained Model

The model was trained and saved in Notebook 41. We load it instead of training it again.


In [3]:
model = joblib.load(MODEL_PATH)

print("Saved model loaded successfully.")
print("Model type:", type(model).__name__)
print("Number of trees:", model.n_estimators)


Saved model loaded successfully.
Model type: RandomForestClassifier
Number of trees: 50


## Part 4 — Recreate the Test Set and Measure Accuracy

Notebook 41 used:

- `test_size=0.20`
- `random_state=42`
- `stratify=y`

Using the same settings recreates the same test split, allowing us to verify the saved model's accuracy.


In [5]:
ml_ready_df = pd.read_csv(ML_READY_PATH)

feature_columns = [
    "acc_x", "acc_y", "acc_z",
    "gyro_x", "gyro_y", "gyro_z",
    "acc_magnitude", "gyro_magnitude"
]

required_columns = feature_columns + ["activity"]
missing_columns = [
    column for column in required_columns
    if column not in ml_ready_df.columns
]

if missing_columns:
    raise ValueError(
        "The processed dataset is missing required columns: "
        + ", ".join(missing_columns)
    )

X = ml_ready_df[feature_columns]
y = ml_ready_df["activity"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

test_predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, test_predictions)

print("=" * 52)
print("        SAVED MODEL PERFORMANCE")
print("=" * 52)
print(f"Testing samples : {len(X_test):,}")
print(f"Correct results : {(test_predictions == y_test).sum():,}")
print(f"Model accuracy  : {accuracy:.2%}")
print("=" * 52)


        SAVED MODEL PERFORMANCE
Testing samples : 864
Correct results : 856
Model accuracy  : 99.07%


## Part 5 — Create 10 New Sensor Samples

Each sample includes:

- `acc_x`, `acc_y`, `acc_z`: acceleration along three axes
- `gyro_x`, `gyro_y`, `gyro_z`: rotation along three axes

The values below are intentionally different so that the model receives a variety of motion patterns.

You may replace any values and rerun the prediction cells.


In [12]:
new_sensor_values = pd.DataFrame([
    {"sample_id": 1,  "acc_x":  0.12, "acc_y":  0.08, "acc_z":  0.96, "gyro_x":  0.01, "gyro_y":  0.02, "gyro_z":  0.01},
    {"sample_id": 2,  "acc_x":  0.45, "acc_y": -0.22, "acc_z":  0.81, "gyro_x":  0.16, "gyro_y": -0.08, "gyro_z":  0.12},
    {"sample_id": 3,  "acc_x":  1.18, "acc_y":  0.64, "acc_z":  1.42, "gyro_x":  0.72, "gyro_y":  0.45, "gyro_z":  0.61},
    {"sample_id": 4,  "acc_x": -0.08, "acc_y":  0.04, "acc_z":  1.01, "gyro_x":  0.00, "gyro_y":  0.01, "gyro_z": -0.01},
    {"sample_id": 5,  "acc_x":  0.68, "acc_y":  0.31, "acc_z":  0.74, "gyro_x":  0.28, "gyro_y":  0.19, "gyro_z":  0.24},
    {"sample_id": 6,  "acc_x": -0.42, "acc_y":  0.55, "acc_z":  0.63, "gyro_x": -0.21, "gyro_y":  0.34, "gyro_z":  0.18},
    {"sample_id": 7,  "acc_x":  1.62, "acc_y": -0.88, "acc_z":  1.27, "gyro_x":  0.95, "gyro_y": -0.57, "gyro_z":  0.83},
    {"sample_id": 8,  "acc_x":  0.03, "acc_y": -0.02, "acc_z":  0.99, "gyro_x":  0.01, "gyro_y": -0.01, "gyro_z":  0.00},
    {"sample_id": 9,  "acc_x": -0.76, "acc_y": -0.38, "acc_z":  0.92, "gyro_x": -0.44, "gyro_y": -0.29, "gyro_z":  0.37},
    {"sample_id": 10, "acc_x":  0.91, "acc_y":  0.77, "acc_z":  1.05, "gyro_x":  0.53, "gyro_y":  0.48, "gyro_z": -0.32},
    {"sample_id": 11, "acc_x":  0.1, "acc_y":  0.377, "acc_z":  5.05, "gyro_x":  6.53, "gyro_y":  7.48, "gyro_z": -5.32},
])

print("Created", len(new_sensor_values), "new sensor samples.")
display(new_sensor_values)


Created 11 new sensor samples.


,sample_id,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,1,0.12,0.080,0.96,0.01,0.02,0.01
1,2,0.45,-0.220,0.81,0.16,-0.08,0.12
2,3,1.18,0.640,1.42,0.72,0.45,0.61
3,4,-0.08,0.040,1.01,0.00,0.01,-0.01
4,5,0.68,0.310,0.74,0.28,0.19,0.24
5,6,-0.42,0.550,0.63,-0.21,0.34,0.18
6,7,1.62,-0.880,1.27,0.95,-0.57,0.83
7,8,0.03,-0.020,0.99,0.01,-0.01,0.00
8,9,-0.76,-0.380,0.92,-0.44,-0.29,0.37
9,10,0.91,0.770,1.05,0.53,0.48,-0.32


## Part 6 — Engineer the Same Features Used During Training

The saved model expects eight features. Notebook 41 created two additional magnitude features:

```text
acc_magnitude  = √(acc_x² + acc_y² + acc_z²)
gyro_magnitude = √(gyro_x² + gyro_y² + gyro_z²)
```

New data must go through the **same feature-engineering steps** used during training.


In [13]:
prediction_input = new_sensor_values.copy()

prediction_input["acc_magnitude"] = np.sqrt(
    prediction_input["acc_x"] ** 2
    + prediction_input["acc_y"] ** 2
    + prediction_input["acc_z"] ** 2
)

prediction_input["gyro_magnitude"] = np.sqrt(
    prediction_input["gyro_x"] ** 2
    + prediction_input["gyro_y"] ** 2
    + prediction_input["gyro_z"] ** 2
)

print("Magnitude features calculated.")
display(
    prediction_input[
        ["sample_id", "acc_magnitude", "gyro_magnitude"]
    ].round(4)
)


Magnitude features calculated.


,sample_id,acc_magnitude,gyro_magnitude
0,1,0.9708,0.0245
1,2,0.9524,0.2154
2,3,1.9541,1.0455
3,4,1.0140,0.0141
4,5,1.0517,0.4148
5,6,0.9358,0.4383
6,7,2.2387,1.3843
7,8,0.9907,0.0141
8,9,1.2524,0.6439
9,10,1.5886,0.7834


## Part 7 — Predict the Activity for All 10 Samples

The saved model receives the columns in the exact same order used during training.


In [14]:
model_input = prediction_input[feature_columns]

predicted_activities = model.predict(model_input)

# Random Forest can also estimate a probability for each possible activity.
prediction_probabilities = model.predict_proba(model_input)
prediction_confidence = prediction_probabilities.max(axis=1)

results = prediction_input.copy()
results["predicted_activity"] = predicted_activities
results["confidence"] = prediction_confidence

print("Predictions completed for all 10 samples.")


Predictions completed for all 10 samples.


## Part 8 — Display the Prediction Table

The final table includes:

- all six input sensor values
- both engineered magnitude values
- the predicted activity
- the model's confidence for that prediction

> Confidence is useful, but it should not be interpreted as a guarantee that the prediction is correct.


In [15]:
display_columns = [
    "sample_id",
    "acc_x", "acc_y", "acc_z",
    "gyro_x", "gyro_y", "gyro_z",
    "acc_magnitude", "gyro_magnitude",
    "predicted_activity", "confidence"
]

formatted_results = results[display_columns].copy()

styled_results = (
    formatted_results.style
    .format({
        "acc_x": "{:.3f}",
        "acc_y": "{:.3f}",
        "acc_z": "{:.3f}",
        "gyro_x": "{:.3f}",
        "gyro_y": "{:.3f}",
        "gyro_z": "{:.3f}",
        "acc_magnitude": "{:.3f}",
        "gyro_magnitude": "{:.3f}",
        "confidence": "{:.1%}",
    })
    .set_caption(
        f"MotionSense-AI Predictions for 10 New Sensor Samples "
        f"| Saved Model Accuracy: {accuracy:.2%}"
    )
    .set_properties(**{
        "text-align": "center",
        "padding": "8px",
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "17px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "10px 0"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-weight", "bold"),
                ("padding", "8px"),
            ],
        },
    ])
)

display(styled_results)


,sample_id,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_magnitude,gyro_magnitude,predicted_activity,confidence
0,1,0.120,0.080,0.960,0.010,0.020,0.010,0.971,0.024,Laying,46.0%
1,2,0.450,-0.220,0.810,0.160,-0.080,0.120,0.952,0.215,Walking,36.0%
2,3,1.180,0.640,1.420,0.720,0.450,0.610,1.954,1.045,Walking Downstairs,44.0%
3,4,-0.080,0.040,1.010,0.000,0.010,-0.010,1.014,0.014,Laying,44.0%
4,5,0.680,0.310,0.740,0.280,0.190,0.240,1.052,0.415,Walking,48.0%
5,6,-0.420,0.550,0.630,-0.210,0.340,0.180,0.936,0.438,Walking,44.0%
6,7,1.620,-0.880,1.270,0.950,-0.570,0.830,2.239,1.384,Walking Downstairs,60.0%
7,8,0.030,-0.020,0.990,0.010,-0.010,0.000,0.991,0.014,Laying,42.0%
8,9,-0.760,-0.380,0.920,-0.440,-0.290,0.370,1.252,0.644,Walking Downstairs,66.0%
9,10,0.910,0.770,1.050,0.530,0.480,-0.320,1.589,0.783,Walking Downstairs,42.0%


## Part 9 — Print a Clear Prediction Summary

This cell prints one readable sentence for each sample.


In [16]:
print("=" * 72)
print(f"MOTIONSENSE-AI — MODEL ACCURACY: {accuracy:.2%}")
print("=" * 72)

for row in results.itertuples(index=False):
    print(
        f"Sample {row.sample_id:>2}: "
        f"Predicted Activity = {row.predicted_activity:<12} "
        f"| Confidence = {row.confidence:.1%}"
    )

print("=" * 72)


MOTIONSENSE-AI — MODEL ACCURACY: 99.07%
Sample  1: Predicted Activity = Laying       | Confidence = 46.0%
Sample  2: Predicted Activity = Walking      | Confidence = 36.0%
Sample  3: Predicted Activity = Walking Downstairs | Confidence = 44.0%
Sample  4: Predicted Activity = Laying       | Confidence = 44.0%
Sample  5: Predicted Activity = Walking      | Confidence = 48.0%
Sample  6: Predicted Activity = Walking      | Confidence = 44.0%
Sample  7: Predicted Activity = Walking Downstairs | Confidence = 60.0%
Sample  8: Predicted Activity = Laying       | Confidence = 42.0%
Sample  9: Predicted Activity = Walking Downstairs | Confidence = 66.0%
Sample 10: Predicted Activity = Walking Downstairs | Confidence = 42.0%
Sample 11: Predicted Activity = Walking Downstairs | Confidence = 46.0%


## Part 10 — Optional: Save the Prediction Results

The results can be exported to a CSV file for a report, website, or dashboard.


In [17]:
OUTPUT_PATH = PROCESSED_DIR / "motionsense_10_new_predictions.csv"

formatted_results.to_csv(OUTPUT_PATH, index=False)

print("Prediction results saved to:", OUTPUT_PATH)


Prediction results saved to: ..\data\processed\motionsense_10_new_predictions.csv


## Final Takeaways

1. A trained model can be saved and reused without training it again.
2. New data must contain the same features used during training.
3. Feature engineering must be repeated exactly for new sensor samples.
4. Scikit-learn expects the feature columns in the same structure and order.
5. The model returns one predicted activity for every input row.
6. Accuracy measures performance against labeled test data.
7. Confidence indicates how strongly the trees agreed, but it does not prove that a prediction is correct.

## Big Idea

```text
Saved Model + Correctly Prepared New Data → New Activity Predictions
```
